# PixelClear Training Notebook
## Complete Training Pipeline with Hyperparameter Support

This notebook provides:
1. Data loading and preprocessing
2. Model training with configurable hyperparameters
3. Validation and metrics tracking
4. Checkpoint saving/loading
5. Visualization of results

**All code is self-contained - no external imports needed!**

## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Optional, List, Tuple
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from tqdm import tqdm
import csv
import random
import math
from PIL import Image

BASE_DIR = Path("/Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 2. Configuration

**Modify these settings to customize your training:**

In [ ]:
# Model Configuration
MODEL_CONFIG = {
    'in_channels': 3,
    'base_channels': 32,  # Try: 16, 24, 32, 48
    'num_fusion_blocks': 4,  # Try: 2, 3, 4, 6
    'use_transformer': True,
    'use_compression': True,
    'use_lowlight': True,
    'img_size': 256
}

# Training Configuration
TRAINING_CONFIG = {
    'epochs': 10,  # Start with 5-10 for testing, then use 100
    'batch_size': 8,  # Try: 4, 8, 16 (reduce if out of memory)
    'learning_rate': 1e-4,  # Try: 1e-5, 5e-5, 1e-4, 5e-4
    'weight_decay': 1e-4,
    'loss_type': 'combined',  # Options: 'l1', 'l2', 'mse', 'combined'
    'l1_weight': 1.0,
    'mse_weight': 0.1,
    'optimizer': 'adamw',  # Options: 'adam', 'adamw', 'sgd'
    'scheduler': 'cosine',  # Options: 'step', 'cosine', 'plateau', None
    'grad_clip': 1.0,
    'log_interval': 100
}

# Data Configuration
DATA_CONFIG = {
    'train_csv': BASE_DIR / 'indices/gopro_train.csv',
    'val_csv': BASE_DIR / 'indices/gopro_test.csv',
    'patch_size': 256,
    'num_workers': 0,  # Use 0 if on Windows or having issues
    'adaptive_sampling': True,
    'num_candidates': 5
}

# Save Configuration
SAVE_CONFIG = {
    'save_dir': BASE_DIR / 'experiments/notebook_training',
    'save_interval': 10,  # Save checkpoint every N epochs
    'save_best': True  # Save best model based on validation PSNR
}

print("Configuration loaded!")

## 3. Data Preprocessing Components (Self-contained)

In [ ]:
# Data preprocessing functions
def compute_laplacian_variance(img_tensor: torch.Tensor) -> float:
    if img_tensor.dim() == 3:
        gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
    else:
        gray = img_tensor
    laplacian_kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32)
    laplacian_kernel = laplacian_kernel.view(1, 1, 3, 3)
    gray_4d = gray.unsqueeze(0).unsqueeze(0)
    laplacian = F.conv2d(gray_4d, laplacian_kernel, padding=1)
    return torch.var(laplacian).item()

class AdaptivePatchSampler:
    def __init__(self, patch_size: int = 256, num_candidates: int = 5, mode: str = "train"):
        self.patch_size = patch_size
        self.num_candidates = num_candidates
        self.mode = mode
    def sample_patch(self, input_img, target_img):
        w, h = input_img.size
        ps = self.patch_size
        if w < ps or h < ps:
            return input_img, target_img
        if self.mode == "train" and self.num_candidates > 1:
            candidates = []
            for _ in range(self.num_candidates):
                left = random.randint(0, w - ps)
                top = random.randint(0, h - ps)
                patch_input = TF.crop(input_img, top, left, ps, ps)
                patch_tensor = TF.to_tensor(patch_input)
                blur_score = compute_laplacian_variance(patch_tensor)
                candidates.append((left, top, blur_score))
            candidates.sort(key=lambda x: x[2])
            left, top, _ = candidates[0]
        else:
            left = random.randint(0, w - ps) if w > ps else 0
            top = random.randint(0, h - ps) if h > ps else 0
        input_patch = TF.crop(input_img, top, left, ps, ps)
        target_patch = TF.crop(target_img, top, left, ps, ps)
        return input_patch, target_patch

class UnifiedRestorationDataset(Dataset):
    def __init__(self, index_csv: str, patch_size: int = 256, mode: str = "train", normalize: Optional[Dict] = None, adaptive_sampling: bool = True, num_candidates: int = 5):
        self.index_csv = Path(index_csv)
        self.patch_size = patch_size
        self.mode = mode
        self.normalize = normalize
        self.adaptive_sampling = adaptive_sampling and mode == "train"
        self.pairs = self._load_index()
        if self.adaptive_sampling:
            self.patch_sampler = AdaptivePatchSampler(patch_size, num_candidates, mode)
        else:
            self.patch_sampler = AdaptivePatchSampler(patch_size, 1, mode)
    def _load_index(self):
        pairs = []
        with open(self.index_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                pairs.append(row)
        return pairs
    def _load_image(self, path: str):
        img = Image.open(path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return img
    def _augment_pair(self, input_img, target_img):
        if self.mode != "train":
            return input_img, target_img
        if random.random() < 0.5:
            input_img = TF.hflip(input_img)
            target_img = TF.hflip(target_img)
        if random.random() < 0.5:
            input_img = TF.vflip(input_img)
            target_img = TF.vflip(target_img)
        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            input_img = TF.rotate(input_img, angle)
            target_img = TF.rotate(target_img, angle)
        return input_img, target_img
    def _to_tensor_normalize(self, img):
        tensor = TF.to_tensor(img)
        if self.normalize:
            mean = torch.tensor(self.normalize['mean']).view(3, 1, 1)
            std = torch.tensor(self.normalize['std']).view(3, 1, 1)
            tensor = (tensor - mean) / std
        return tensor
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        input_img = self._load_image(pair['input_path'])
        target_img = self._load_image(pair['target_path'])
        if self.patch_size:
            input_img, target_img = self.patch_sampler.sample_patch(input_img, target_img)
        input_img, target_img = self._augment_pair(input_img, target_img)
        input_tensor = self._to_tensor_normalize(input_img)
        target_tensor = self._to_tensor_normalize(target_img)
        return {'input': input_tensor, 'target': target_tensor, 'dataset': pair.get('dataset', 'unknown'), 'scene': pair.get('scene', 'unknown')}

def create_dataloader(index_csv: str, batch_size: int = 8, patch_size: int = 256, mode: str = "train", normalize: Optional[Dict] = None, num_workers: int = 4, adaptive_sampling: bool = True, num_candidates: int = 5):
    dataset = UnifiedRestorationDataset(index_csv=index_csv, patch_size=patch_size, mode=mode, normalize=normalize, adaptive_sampling=adaptive_sampling, num_candidates=num_candidates)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=(mode == "train"), num_workers=num_workers, pin_memory=True, drop_last=(mode == "train"))
    return loader

print("✓ Data preprocessing components defined")


## 4. Model Architecture (Self-contained)

**Note:** All model components are included here. You can also import from notebook 03 if needed.

## 1. CNN Components

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return x

class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = self.sigmoid(avg_out + max_out)
        return x * out

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(combined))
        return x * attention

class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, kernel_size: int = 7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

class LightweightResidualBlock(nn.Module):
    def __init__(self, channels: int, use_attention: bool = True):
        super().__init__()
        self.use_attention = use_attention
        self.conv1 = DepthwiseSeparableConv(channels, channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = DepthwiseSeparableConv(channels, channels)
        if use_attention:
            self.attention = CBAM(channels, reduction=8)
        self.relu2 = nn.ReLU(inplace=True)
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.conv2(out)
        if self.use_attention:
            out = self.attention(out)
        out = out + residual
        out = self.relu2(out)
        return out

class MultiScaleFeatureExtractor(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        b4 = F.interpolate(b4, size=x.shape[2:], mode='bilinear', align_corners=False)
        out = torch.cat([b1, b2, b3, b4], dim=1)
        return out

print("✓ CNN components defined")

## 2. Transformer Components

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size: int = 256, patch_size: int = 8, in_channels: int = 3, embed_dim: int = 64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False),
            nn.BatchNorm2d(embed_dim)
        )
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x

class LightweightSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, qkv_bias: bool = False, attn_drop: float = 0.0, proj_drop: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, attn

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, mlp_ratio: float = 2.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = LightweightSelfAttention(dim, num_heads=num_heads, attn_drop=drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim), nn.Dropout(drop)
        )
    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights

class GlobalContextTransformer(nn.Module):
    def __init__(self, in_channels: int = 32, embed_dim: int = 64, num_heads: int = 4, num_layers: int = 2, patch_size: int = 8, img_size: int = 256):
        super().__init__()
        self.patch_size = patch_size
        self.img_size = img_size
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads=num_heads, mlp_ratio=2.0, drop=0.0) 
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj_back = nn.Sequential(nn.Linear(embed_dim, in_channels), nn.GELU())
    def forward(self, x, return_attention: bool = False):
        B, C, H, W = x.shape
        x_tokens = self.patch_embed(x)
        x_tokens = x_tokens + self.pos_embed
        attention_maps = []
        for block in self.blocks:
            x_tokens, attn = block(x_tokens)
            if return_attention:
                attention_maps.append(attn)
        x_tokens = self.norm(x_tokens)
        x_tokens = self.proj_back(x_tokens)
        h = w = int(math.sqrt(x_tokens.shape[1]))
        x_out = x_tokens.transpose(1, 2).reshape(B, C, h, w)
        if h != H or w != W:
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)
        if return_attention:
            return x_out, attention_maps
        return x_out

print("✓ Transformer components defined")

## 3. Specialized Degradation Modules

In [ ]:
class CompressionArtifactRemover(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.block_detector = nn.Sequential(
            nn.Conv2d(channels, channels // 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels // 2), nn.ReLU(inplace=True),
            nn.Conv2d(channels // 2, channels, kernel_size=3, padding=1, bias=False), nn.Sigmoid()
        )
        self.smoother = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, x):
        artifact_mask = self.block_detector(x)
        smoothed = self.smoother(x)
        out = x * (1 - artifact_mask) + smoothed * artifact_mask
        return out

class LowLightEnhancer(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.illumination_estimator = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 4, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, kernel_size=1, bias=False), nn.Sigmoid()
        )
        self.enhancer = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, x):
        illumination = self.illumination_estimator(x)
        enhanced = self.enhancer(x)
        out = x + enhanced * (1 - illumination)
        return out

print("✓ Specialized modules defined")

## 4. Fusion Architecture

In [ ]:
class FusionBlock(nn.Module):
    def __init__(self, channels: int, use_transformer: bool = True, use_compression: bool = True, use_lowlight: bool = True):
        super().__init__()
        self.use_transformer = use_transformer
        self.use_compression = use_compression
        self.use_lowlight = use_lowlight
        self.cnn_block = LightweightResidualBlock(channels, use_attention=True)
        if use_transformer:
            self.transformer = GlobalContextTransformer(in_channels=channels, embed_dim=channels * 2, num_heads=4, num_layers=1, patch_size=8)
        if use_compression:
            self.compression_module = CompressionArtifactRemover(channels)
        if use_lowlight:
            self.lowlight_module = LowLightEnhancer(channels)
        self.fusion_gate = nn.Sequential(
            nn.Conv2d(channels * 2 if use_transformer else channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels), nn.Sigmoid()
        )
        self.output_conv = nn.Sequential(
            nn.Conv2d(channels * 2 if use_transformer else channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, return_attention: bool = False):
        cnn_feat = self.cnn_block(x)
        if self.use_compression:
            cnn_feat = self.compression_module(cnn_feat)
        if self.use_lowlight:
            cnn_feat = self.lowlight_module(cnn_feat)
        if self.use_transformer:
            transformer_feat, attention_maps = self.transformer(x, return_attention=True)
            combined = torch.cat([cnn_feat, transformer_feat], dim=1)
            fusion_weight = self.fusion_gate(combined)
            out = self.output_conv(combined)
            if return_attention:
                return out, attention_maps
            return out
        else:
            return cnn_feat

class PixelClearFusionNet(nn.Module):
    def __init__(self, in_channels: int = 3, base_channels: int = 32, num_fusion_blocks: int = 4, use_transformer: bool = True, use_compression: bool = True, use_lowlight: bool = True, img_size: int = 256):
        super().__init__()
        self.use_transformer = use_transformer
        self.input_conv = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True)
        )
        self.multi_scale_extractor = MultiScaleFeatureExtractor(base_channels, base_channels)
        self.fusion_blocks = nn.ModuleList([
            FusionBlock(channels=base_channels, use_transformer=use_transformer and (i % 2 == 0), use_compression=use_compression, use_lowlight=use_lowlight)
            for i in range(num_fusion_blocks)
        ])
        self.output_conv = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)
        )
    def forward(self, x, return_features: bool = False, return_attention: bool = False):
        residual = x
        feat = self.input_conv(x)
        feat = self.multi_scale_extractor(feat)
        intermediate_features = [feat]
        all_attention_maps = []
        for block in self.fusion_blocks:
            if return_attention and self.use_transformer:
                feat, attn_maps = block(feat, return_attention=True)
                if attn_maps:  # Only extend if not empty
                    all_attention_maps.extend(attn_maps)
            else:
                feat = block(feat)
            intermediate_features.append(feat)
        out = self.output_conv(feat)
        out = out + residual
        if return_features and return_attention:
            return out, intermediate_features, all_attention_maps
        elif return_features:
            return out, intermediate_features
        elif return_attention:
            return out, all_attention_maps
        else:
            return out

print("✓ Fusion architecture defined")

## 5. Enhanced Explainability Module

In [ ]:
class EnhancedExplainabilityModule(nn.Module):
    def __init__(self):
        super().__init__()
    def generate_pixel_change_map(self, input_img: torch.Tensor, output_img: torch.Tensor) -> torch.Tensor:
        diff_r = torch.abs(output_img[:, 0:1] - input_img[:, 0:1])
        diff_g = torch.abs(output_img[:, 1:2] - input_img[:, 1:2])
        diff_b = torch.abs(output_img[:, 2:3] - input_img[:, 2:3])
        change_map = (diff_r + diff_g + diff_b) / 3.0
        change_map = (change_map - change_map.min()) / (change_map.max() - change_map.min() + 1e-8)
        return change_map
    def generate_improvement_map(self, input_img: torch.Tensor, output_img: torch.Tensor, target_img: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        def laplacian_variance(img):
            gray = 0.299 * img[:, 0] + 0.587 * img[:, 1] + 0.114 * img[:, 2]
            laplacian_kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32, device=img.device)
            laplacian_kernel = laplacian_kernel.view(1, 1, 3, 3)
            gray_4d = gray.unsqueeze(1)
            laplacian = F.conv2d(gray_4d, laplacian_kernel, padding=1)
            return torch.var(laplacian, dim=[2, 3], keepdim=True)
        input_sharpness = laplacian_variance(input_img)
        output_sharpness = laplacian_variance(output_img)
        sharpness_improvement = (output_sharpness - input_sharpness) / (input_sharpness + 1e-8)
        input_brightness = torch.mean(input_img, dim=1, keepdim=True)
        output_brightness = torch.mean(output_img, dim=1, keepdim=True)
        brightness_improvement = output_brightness - input_brightness
        change_map = self.generate_pixel_change_map(input_img, output_img)
        results = {'change_map': change_map, 'sharpness_improvement': sharpness_improvement, 'brightness_improvement': brightness_improvement}
        if target_img is not None:
            error_map = torch.mean(torch.abs(output_img - target_img), dim=1, keepdim=True)
            results['error_map'] = error_map
        return results
    def generate_transformer_attention_visualization(self, attention_maps: List[torch.Tensor], img_size: int = 256, patch_size: int = 8) -> torch.Tensor:
        if not attention_maps:
            return None
        avg_attention = torch.stack([attn.mean(dim=1).mean(dim=1) for attn in attention_maps]).mean(dim=0)
        h = w = int(math.sqrt(avg_attention.shape[-1]))
        attention_spatial = avg_attention.reshape(-1, h, w)
        attention_spatial = attention_spatial.unsqueeze(1)
        attention_spatial = F.interpolate(attention_spatial, size=(img_size, img_size), mode='bilinear', align_corners=False)
        return attention_spatial.squeeze(1)
    def forward(self, input_img: torch.Tensor, output_img: torch.Tensor, features: Optional[List[torch.Tensor]] = None, attention_maps: Optional[List[torch.Tensor]] = None, target_img: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        results = self.generate_improvement_map(input_img, output_img, target_img)
        if attention_maps:
            transformer_attn = self.generate_transformer_attention_visualization(attention_maps, img_size=input_img.shape[-1])
            if transformer_attn is not None:
                results['transformer_attention'] = transformer_attn
        if features:
            feature_attention = torch.mean(features[-1], dim=1, keepdim=True)
            feature_attention = (feature_attention - feature_attention.min()) / (feature_attention.max() - feature_attention.min() + 1e-8)
            results['feature_attention'] = feature_attention
        return results

print("✓ Explainability module defined")

## 5. Training Functions

In [ ]:
# Loss functions
def compute_loss(pred, target, loss_type='combined', l1_weight=1.0, mse_weight=0.1):
    if loss_type == 'l1':
        return F.l1_loss(pred, target)
    elif loss_type == 'l2':
        return F.mse_loss(pred, target)
    elif loss_type == 'mse':
        return F.mse_loss(pred, target)
    elif loss_type == 'combined':
        l1 = F.l1_loss(pred, target)
        mse = F.mse_loss(pred, target)
        return l1_weight * l1 + mse_weight * mse
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

# Metrics
def compute_psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(torch.tensor(max_val)) / torch.log10(torch.tensor(10.0)) - 10 * torch.log10(mse) / torch.log10(torch.tensor(10.0))

def compute_ssim(pred, target):
    # Simplified SSIM (for full implementation, use pytorch-msssim)
    mu1 = pred.mean()
    mu2 = target.mean()
    sigma1_sq = pred.var()
    sigma2_sq = target.var()
    sigma12 = ((pred - mu1) * (target - mu2)).mean()
    c1, c2 = 0.01**2, 0.03**2
    ssim = ((2*mu1*mu2 + c1) * (2*sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim

print("✓ Training functions defined")

## 6. Create Model and Data Loaders

In [ ]:
# Create model
model = PixelClearFusionNet(**MODEL_CONFIG).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created: {total_params:,} parameters")
print(f"Model size: ~{total_params * 4 / (1024**2):.2f} MB")

# Create data loaders
train_loader = create_dataloader(
    index_csv=str(DATA_CONFIG['train_csv']),
    batch_size=TRAINING_CONFIG['batch_size'],
    patch_size=DATA_CONFIG['patch_size'],
    mode='train',
    num_workers=DATA_CONFIG['num_workers'],
    adaptive_sampling=DATA_CONFIG['adaptive_sampling'],
    num_candidates=DATA_CONFIG['num_candidates']
)

val_loader = create_dataloader(
    index_csv=str(DATA_CONFIG['val_csv']),
    batch_size=TRAINING_CONFIG['batch_size'],
    patch_size=DATA_CONFIG['patch_size'],
    mode='val',
    num_workers=DATA_CONFIG['num_workers'],
    adaptive_sampling=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 7. Setup Optimizer and Scheduler

In [ ]:
# Optimizer
if TRAINING_CONFIG['optimizer'] == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
elif TRAINING_CONFIG['optimizer'] == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
elif TRAINING_CONFIG['optimizer'] == 'sgd':
    optimizer = optim.SGD(model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'], momentum=0.9)
else:
    raise ValueError(f"Unknown optimizer: {TRAINING_CONFIG['optimizer']}")

# Scheduler
scheduler = None
if TRAINING_CONFIG['scheduler'] == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAINING_CONFIG['epochs'])
elif TRAINING_CONFIG['scheduler'] == 'step':
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
elif TRAINING_CONFIG['scheduler'] == 'plateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

print(f"Optimizer: {TRAINING_CONFIG['optimizer']}")
print(f"Scheduler: {TRAINING_CONFIG['scheduler']}")

## 8. Training Loop

In [ ]:
# Create save directory
SAVE_CONFIG['save_dir'].mkdir(parents=True, exist_ok=True)

# Training history
history = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}
best_psnr = 0.0

# Training loop
for epoch in range(1, TRAINING_CONFIG['epochs'] + 1):
    # Training phase
    model.train()
    train_loss = 0.0
    train_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{TRAINING_CONFIG['epochs']}")
    for batch_idx, batch in enumerate(pbar):
        input_img = batch['input'].to(DEVICE)
        target_img = batch['target'].to(DEVICE)
        
        optimizer.zero_grad()
        output = model(input_img)
        loss = compute_loss(output, target_img, TRAINING_CONFIG['loss_type'], TRAINING_CONFIG['l1_weight'], TRAINING_CONFIG['mse_weight'])
        loss.backward()
        
        if TRAINING_CONFIG['grad_clip'] > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAINING_CONFIG['grad_clip'])
        
        optimizer.step()
        
        train_loss += loss.item()
        train_batches += 1
        
        if batch_idx % TRAINING_CONFIG['log_interval'] == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / train_batches
    history['train_loss'].append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0
    val_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_img = batch['input'].to(DEVICE)
            target_img = batch['target'].to(DEVICE)
            
            output = model(input_img)
            loss = compute_loss(output, target_img, TRAINING_CONFIG['loss_type'], TRAINING_CONFIG['l1_weight'], TRAINING_CONFIG['mse_weight'])
            
            val_loss += loss.item()
            val_psnr += compute_psnr(output, target_img).item()
            val_ssim += compute_ssim(output, target_img).item()
            val_batches += 1
    
    avg_val_loss = val_loss / val_batches
    avg_val_psnr = val_psnr / val_batches
    avg_val_ssim = val_ssim / val_batches
    
    history['val_loss'].append(avg_val_loss)
    history['val_psnr'].append(avg_val_psnr)
    history['val_ssim'].append(avg_val_ssim)
    
    # Update scheduler
    if scheduler:
        if TRAINING_CONFIG['scheduler'] == 'plateau':
            scheduler.step(avg_val_psnr)
        else:
            scheduler.step()
    
    # Print epoch summary
    print(f"\nEpoch {epoch}:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Val PSNR: {avg_val_psnr:.2f} dB")
    print(f"  Val SSIM: {avg_val_ssim:.4f}")
    
    # Save checkpoint
    if epoch % SAVE_CONFIG['save_interval'] == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'config': {'model': MODEL_CONFIG, 'training': TRAINING_CONFIG}
        }
        torch.save(checkpoint, SAVE_CONFIG['save_dir'] / f'checkpoint_epoch_{epoch}.pth')
        print(f"  Saved checkpoint: epoch_{epoch}.pth")
    
    # Save best model
    if SAVE_CONFIG['save_best'] and avg_val_psnr > best_psnr:
        best_psnr = avg_val_psnr
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'val_psnr': avg_val_psnr,
            'config': {'model': MODEL_CONFIG, 'training': TRAINING_CONFIG}
        }
        torch.save(checkpoint, SAVE_CONFIG['save_dir'] / 'best_model.pth')
        print(f"  Saved best model (PSNR: {best_psnr:.2f} dB)")

print("\n✓ Training completed!")